In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret found; cloning this PRIVATE repo will fail.\n"
                  "Add a GitHub token (repo scope) via the key icon (Secrets) as 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok
                else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_root)], check=True)
        subprocess.run(["git", "-C", str(_root), "remote", "set-url", "origin",
                        f"https://github.com/{_slug}.git"])  # keep the token out of the saved remote
    os.chdir(_root / "07-application-agent-framework/agent-fundamentals/mistral-agent-core/solutions")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


In [ ]:
# make `import agentcore` work from anywhere (auto-inserted)
import sys, pathlib
_r = pathlib.Path.cwd().resolve()
while _r != _r.parent and not (_r / 'agentcore').exists():
    _r = _r.parent
sys.path.insert(0, str(_r))

# 05 · Going live on Mistral

The loop you built in notebooks 01–04 never mentions a provider. It only needs
something with a `.generate(messages, tools) -> Response` method. `FakeLLM` is that
for offline practice; `MistralLLM` is that for Mistral's API. **Swapping one for the
other is one line** — that is the whole point of keeping the loop provider-agnostic.

This notebook shows the swap and the small amount of translation the adapter does
between our shapes and Mistral's. It runs **offline**: we exercise the pure
conversion functions and drive the loop with a fake Mistral-shaped client. A final,
guarded cell calls the real API only if you have set `MISTRAL_API_KEY`.

In [ ]:
import json
from agentcore import Agent, tool
from agentcore.mistral_llm import parse_response, to_mistral_messages, to_mistral_tools

## 1. Tools → Mistral's function shape
Our tool schema is `{"name","description","parameters"}`. Mistral (like the OpenAI
shape) wants each tool wrapped as `{"type":"function","function": {...}}`.

In [ ]:
@tool
def get_balance(account_id: str) -> dict:
    """Return the balance for an account."""
    return {"account_id": account_id, "balance": 1234.5}

print(json.dumps(to_mistral_tools([get_balance.schema]), indent=2))

## 2. Messages → Mistral's format
System, user and tool messages already match Mistral. Only our **assistant tool
calls** need reshaping: we carry `args` as a dict, Mistral wants
`function.arguments` as a JSON *string*.

In [ ]:
demo = [
    {"role": "user", "content": "balance for a1?"},
    {"role": "assistant", "content": "", "tool_calls": [{"id": "c1", "name": "get_balance", "args": {"account_id": "a1"}}]},
    {"role": "tool", "name": "get_balance", "tool_call_id": "c1", "content": '{"balance": 1234.5}'},
]
converted = to_mistral_messages(demo)
print("assistant tool call becomes:", json.dumps(converted[1]["tool_calls"][0], indent=2))

## 3. Mistral's reply → our `Response`
A Mistral reply is `response.choices[0].message` with `.content` and `.tool_calls`
(each `tool_call` has `.id` and `.function.name` / `.function.arguments`). The
adapter turns that back into the `Response` the loop understands.

In [ ]:
fake_reply = {"choices": [{"message": {"content": "", "tool_calls": [
    {"id": "call_1", "function": {"name": "get_balance", "arguments": '{"account_id": "a1"}'}}]}}]}
r = parse_response(fake_reply)
print("parsed →", [(tc.name, tc.args, tc.id) for tc in r.tool_calls])

## 4. The whole loop, offline, against a Mistral-shaped client
To prove the swap works without spending a token, here is a stand-in that returns
Mistral-shaped dicts. Notice `Agent` is unchanged from notebook 04 — only the model
object differs.

In [ ]:
class FakeMistralClient:
    """Returns Mistral-shaped replies from a script (stands in for the real API)."""
    model_name = "mistral-large-latest"
    def __init__(self, replies): self.replies, self.i = replies, 0
    def generate(self, messages, tools=None):
        reply = self.replies[self.i]; self.i += 1
        return parse_response(reply)

replies = [
    {"choices": [{"message": {"content": "", "tool_calls": [
        {"id": "c1", "function": {"name": "get_balance", "arguments": '{"account_id": "a1"}'}}]}}]},
    {"choices": [{"message": {"content": "Your balance is SGD 1,234.50.", "tool_calls": None}}]},
]
result = Agent(FakeMistralClient(replies), tools=[get_balance]).run("what's my balance on a1?")
print(result.transcript())
print("\nanswer:", result.text)

## Exercise 5.1 — implement the tool converter
Without calling the library's `to_mistral_tools`, write `my_to_mistral_tools(schemas)`
that wraps each of our tool schemas as `{"type": "function", "function": <schema>}`,
returning `None` for an empty or missing list.

In [ ]:
def my_to_mistral_tools(schemas):
    if not schemas:
        return None
    return [{"type": "function", "function": s} for s in schemas]

In [ ]:
assert my_to_mistral_tools([get_balance.schema]) == to_mistral_tools([get_balance.schema])
assert my_to_mistral_tools([]) is None
print("✅ tool converter matches the library")

## Exercise 5.2 — parse a Mistral tool-call reply
Write `first_tool_call(reply)` that, given a Mistral-shaped reply dict, returns
`(name, args_dict)` for the first tool call — parsing the JSON `arguments` string.
(This is the heart of what `parse_response` does.)

In [ ]:
def first_tool_call(reply):
    tc = reply["choices"][0]["message"]["tool_calls"][0]
    return tc["function"]["name"], json.loads(tc["function"]["arguments"])

In [ ]:
name, args = first_tool_call(fake_reply)
assert name == "get_balance" and args == {"account_id": "a1"}
print("✅ parsed the tool call:", name, args)

## Exercise 5.3 — pick the right model
Part of the Mistral deployment story is choosing the *cheapest model that clears the
bar*. Map each need to a model string (see `docs/MISTRAL.md`): fill in `CHOICES`.

* `"agentic"`   → the flagship, best at multi-step tool use
* `"reasoning"` → step-by-step reasoning
* `"cheap_edge"`→ open-weight, runs on-device / cheapest
* `"coding"`    → code generation and autocomplete

In [ ]:
CHOICES = {}
CHOICES = {
    "agentic": "mistral-large-latest",
    "reasoning": "magistral-medium-latest",
    "cheap_edge": "ministral-8b-latest",
    "coding": "codestral-latest",
}

In [ ]:
assert CHOICES["agentic"] == "mistral-large-latest"
assert CHOICES["reasoning"].startswith("magistral")
assert CHOICES["cheap_edge"].startswith("ministral")
assert CHOICES["coding"].startswith("codestral")
print("✅ model choices:", CHOICES)

## 5. Run it for real (only if you have a key)
This cell calls the live API when `MISTRAL_API_KEY` is set, and otherwise prints how
to enable it — so the notebook still runs clean in CI and offline.

In [ ]:
import os
if os.environ.get("MISTRAL_API_KEY"):
    from agentcore.mistral_llm import MistralLLM
    agent = Agent(MistralLLM(model="mistral-large-latest"), tools=[get_balance],
                  instruction="You are a bank assistant. Use tools for every fact.")
    live = agent.run("what's the balance on account a1?")
    print(live.transcript())
    print("\nlive answer:", live.text)
else:
    print("No MISTRAL_API_KEY set — skipping the live call.")
    print("To run it: pip install mistralai ; export MISTRAL_API_KEY=... ; re-run this cell.")

## The one-minute version (Mistral)
The deployment-strategist question is rarely "which model is smartest" — it is *which
model clears the bar at the lowest cost and the right deployment posture*. Mistral's
edge is that the small models are **open-weight (Apache-2.0)**, so the same agent can
run in a customer's VPC or on-prem for data-sovereignty reasons — a real differentiator
in regulated APAC accounts. Be ready to say: start on `la Plateforme` with
`mistral-large` for the hard agent, route routine turns to `mistral-small`, reach for
`magistral` only where reasoning pays, and offer a self-hosted `small`/`ministral`
path when the data cannot leave the customer's environment.